In [1]:
import ssl
import certifi
import urllib.request

# Patch SSL to use certifi's certificates instead of Windows store
ssl._create_default_https_context = ssl.create_default_context
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import ee
import geemap
import geopandas as gpd

# ee.Authenticate()
ee.Initialize(project='oceanic-gecko-495018-r2')


In [2]:
# STEP 2: Load the GeoJSON from your local repo folder
# Make sure the file path matches where you saved it in your repo
print("Loading local GeoJSON...")
gdf = gpd.read_file('../../data/INDIA_DISTRICTS.geojson')
print(gdf.columns)


Loading local GeoJSON...
Index(['objectid', 'statecode', 'district', 'state', 'shape_leng',
       'shape_area', 'dist_code', 'D_CODE', 'st_code', 'remarks', 'geometry'],
      dtype='str')


In [3]:
# STEP 3: Filter for the toxic gas chamber states (not MP)
# We are slicing out Punjab and Haryana to lock in our agricultural belt.
agri_belt_gdf = gdf[gdf['state'].isin(['PUNJAB', 'HARYANA'])]

# STEP 4: Convert local geometry to Earth Engine geometry
# This translates your local file into a format Google's servers understand
ee_agri_belt = geemap.geopandas_to_ee(agri_belt_gdf)

print("Target regions locked in. Ready to pull the satellite data.")

Target regions locked in. Ready to pull the satellite data.


In [4]:
# STEP 5: Pulling the Sentinel-2 Satellite Data
# We target the peak stubble burning season (Sept - Nov)
start_date = '2025-09-01'
end_date = '2025-11-30'

# The Cloud Masking function (because clouds = trash data)
def mask_s2_clouds(image):
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus. We want them to be 0 (clear).
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

print("Fetching Sentinel-2 data... this might take a sec bro.")

# Pulling the imagery specifically for your Punjab/Haryana polygon
dataset = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(ee_agri_belt) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds)

# We take the median of all images in that date range to get one clean, cloud-free map
median_image = dataset.median()

# The Magic Math: NDVI = (NIR - Red) / (NIR + Red)
# For Sentinel-2: NIR is Band 8 (B8), Red is Band 4 (B4)
ndvi = median_image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# STEP 6: Visualizing the gas chamber on a map
print("Rendering map...")
Map = geemap.Map()
Map.centerObject(ee_agri_belt, 6) # Zoom level 6 is perfect for a regional view

# Adding the NDVI layer with a dope color palette
# Red = dead/harvested/bare soil, Green = healthy growing crops
ndvi_params = {
    'min': 0.0,
    'max': 1.0,
    'palette': ['red', 'yellow', 'green'] 
}

# Clip the satellite data so it perfectly fits inside the state borders, no spilling over
Map.addLayer(ndvi.clip(ee_agri_belt), ndvi_params, 'NDVI (Plant Health)')

# Add an outline of the states just to make it look professional
Map.addLayer(ee.Image().paint(ee_agri_belt, 0, 2), {'palette': ['black']}, 'Punjab & Haryana Borders')

print("Map generated. Scroll down to look at it.")
Map

Fetching Sentinel-2 data... this might take a sec bro.
Rendering map...
Map generated. Scroll down to look at it.


Map(center=[30.071654782562103, 75.85202480540147], controls=(WidgetControl(options=['position', 'transparent_…

## What does this map represent:

* yellow -> orange -> red: this means the crop is being harvested, like low chlorophyll is observed
* green: this means the crop is standing / there is a forest 

#### what it means for us:
when a farm's patch transitions from *green* to *orange*, the moment it turns *red*, we need to dispatch the trucks in less than 20 days so that the truck reaches the farms before farmer burns the stubble

In [5]:
# STEP 7: Actually saving the data so you don't lose your mind
import os

print("Starting the download... do not close VS Code.")

# We save it as a GeoTIFF file right in your current GitHub folder
out_file = '../../data/punjab_haryana_ndvi.tif'

# Warning: DO NOT change the scale to 10 yet. 
# Sentinel-2 is 10-meter resolution. If you try to download 10m pixels 
# for TWO ENTIRE STATES at once, your laptop will literally melt.
# We use scale=1000 (1km resolution) first just to make sure the pipeline works.

geemap.ee_export_image(
    ndvi.clip(ee_agri_belt), 
    filename=out_file, 
    scale=1000, 
    region=ee_agri_belt.geometry(),
    file_per_band=False
)

print(f"Boom. Saved locally as {out_file}. You can sleep now.")

Starting the download... do not close VS Code.
Generating URL ...
Please wait ...


KeyboardInterrupt: 

In [ ]:
# STEP 8: The "Big Brain" Cropland Mask
print("Summoning the ESA WorldCover dataset...")

# Pull the 2021 ESA WorldCover map
world_cover = ee.ImageCollection("ESA/WorldCover/v200").first()

# In their system, Class 40 = Cropland. 
# We create a binary mask: 1 if it's a farm, 0 if it's anything else.
crop_mask = world_cover.eq(40).clip(ee_agri_belt)

# Now we apply this mask to your existing NDVI map
# This literally erases all non-farm pixels from existence
farm_only_ndvi = ndvi.updateMask(crop_mask)

# Let's visualize the filtered map
Map2 = geemap.Map()
Map2.centerObject(ee_agri_belt, 6)

# The mask itself (showing you where the farms are)
Map2.addLayer(crop_mask.selfMask(), {'palette': ['blue']}, 'Identified Farmlands')

# The NDVI data, but ONLY inside the farms
Map2.addLayer(farm_only_ndvi, ndvi_params, 'NDVI (Farms Only)')

print("Boom. Cities and roads deleted. We only track farms now.")
Map2

Summoning the ESA WorldCover dataset...
Boom. Cities and roads deleted. We only track farms now.


Map(center=[30.071654782562103, 75.85202480540147], controls=(WidgetControl(options=['position', 'transparent_…

earlier it was pulling out data for all parts, now it only does for the agricultural lands in the are (highways, cities... removed)

the ML model we would be building needs to predict when a farm land is going to get harvested, now to predict this if we feed only the data of harvesting season, model would just guess, but instead if we tell it **when the crop is sown** and tell it the **whether data (forecasts and live data)** we can ask it to predict exact time of the farm would harvest the crop <br> 

now to make a good model we need about **4 years of data**, in the 2years we can train the model and use the next two years as validation and testing data (1 year each).<br>

downloading all of that data could make google's servers quit our student plan<br>

so what else can we do:  
* only kharif season (harvested in winter) is the main culprit in pollution
* so we can target only kharif crop data
* 15 may to november end is the target data

15 may: why ?? punjab govt prohibits farmers to sow the land before it for paddy (to help recharge ground water)<br>
30 november: wheat needs to be sown after that <br>

why we need accuracy in predicting the precise date of harvesting: farmers are in hurry, like they need to get rid of the stubble as soon as possible, because they have to prepare their farms for wheat, if our trucks do not reach them in sufficient time, the farmers will not have no choice but to burn the stubble

collecting data from 2022-2025: 15 may to 30 november

In [10]:
# STEP 9: The 4-Year "Padded" ML Dataset Extraction (re added the functions in here so that, it can run alone)

import ee
import geemap
import geopandas as gpd

# 1. Initialize API (Assuming you already did the terminal auth bypass)
ee.Initialize(project='oceanic-gecko-495018-r2')

print("Booting up the AVBE God Cell... don't panic if it takes a minute.")

# 2. Load and Filter the Local GeoJSON for the gas chamber states
# Using your exact local path and the case-insensitive fix
gdf = gpd.read_file('../../data/INDIA_DISTRICTS.geojson')
agri_belt_gdf = gdf[gdf['state'].str.upper().isin(['PUNJAB', 'HARYANA'])]
ee_agri_belt = geemap.geopandas_to_ee(agri_belt_gdf)

# 3. Define the Cloud Masking Function (FIXED)
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    
    # We do the math, but FORCE it to remember its original timestamp
    return image.updateMask(mask).divide(10000).copyProperties(image, ["system:time_start"])

# 4. Generate the ESA WorldCover Crop Mask
world_cover = ee.ImageCollection("ESA/WorldCover/v200").first()
crop_mask = world_cover.eq(40).clip(ee_agri_belt)

# 5. Drop Random Farm Points (Only on actual agricultural land)
print("Dropping 1,000 smart pins on the farms...")
random_points = ee.FeatureCollection.randomPoints(ee_agri_belt.geometry(), 1000, seed=42)
farm_points = crop_mask.reduceRegions(
    collection=random_points, reducer=ee.Reducer.first(), scale=10
).filter(ee.Filter.eq('first', 1))

# 6. The Core Extraction Function
def extract_time_series(image):
    date = image.date().format('YYYY-MM-dd')
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    sampled = ndvi.reduceRegions(collection=farm_points, reducer=ee.Reducer.mean(), scale=10)
    return sampled.map(lambda feature: feature.set('date', date))

# 7. The 4-Year Time Machine Loop (May 15 to Nov 30 only)
def get_yearly_data(year):
    start = ee.Date.fromYMD(year, 5, 15)
    end = ee.Date.fromYMD(year, 11, 30)
    
    yearly_dataset = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(ee_agri_belt) \
        .filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .map(mask_s2_clouds)
        
    return yearly_dataset.map(extract_time_series).flatten()

# 8. Execute the Loop for 2022-2025
print("Crunching 4 years of space data... let him cook.")
years = ee.List([2022, 2023, 2024, 2025])
multi_year_data = ee.FeatureCollection(years.map(get_yearly_data)).flatten()

# 9. Export to Google Drive
export_task = ee.batch.Export.table.toDrive(
    collection=multi_year_data,
    description='AVBE_4Year_Paddy_Training_Data',
    fileFormat='CSV',
    folder='Hackathon_Data' 
)

export_task.start()
print("Boom. Export task sent to the cloud. Go touch grass, your CSV will appear in Google Drive soon.")

Booting up the AVBE God Cell... don't panic if it takes a minute.
Dropping 1,000 smart pins on the farms...
Crunching 4 years of space data... let him cook.
Boom. Export task sent to the cloud. Go touch grass, your CSV will appear in Google Drive soon.


In [12]:
# The fixed status checker that won't crash
tasks = ee.batch.Task.list()

if tasks:
    latest_task = tasks[0]
    status_dict = latest_task.status() # Get the full dictionary
    state = status_dict.get('state', 'UNKNOWN')
    
    print(f"Task: {latest_task.config.get('description', 'Unknown')}")
    print(f"Status: {state} ⏳")
    
    if state == 'COMPLETED':
        print("Boom. Data is ready in your Drive. Time to train the model.")
    elif state == 'FAILED':
        # Safely extract the error message without crashing
        error_msg = status_dict.get('error_message', 'Google refused to elaborate.')
        print(f"Bro... it crashed. Real Error: {error_msg}")
else:
    print("No tasks found.")

Task: AVBE_4Year_Paddy_Training_Data
Status: RUNNING ⏳


In [13]:
"""
gee_extract.py
==============
Google Earth Engine — Raw Data Extraction Script
Harvest Date Prediction Pipeline (Implementation Plan v2)

WHAT THIS SCRIPT DOES
----------------------
1. Authenticates with GEE and initialises the project.
2. Ingests a user-supplied CSV of 1,000 sample points (point_id, lat, lon).
3. Exports THREE separate CSVs to Google Drive (one per satellite source):

   a) sentinel2_raw_<year>.csv  — raw S2 bands (B2,B4,B5,B6,B7,B8,B11,B12) +
                                   SCL cloud mask, per point × date.
   b) sentinel1_raw_<year>.csv  — S1 GRD VH and VV backscatter (dB),
                                   per point × date.
   c) static_layers.csv         — SRTM elevation/slope/aspect + ESA WorldCover
                                   land-cover class, per point (one-time extract).

WHAT THIS SCRIPT DOES NOT DO (by design — see §12, Milestone 0)
-----------------------------------------------------------------
- No temporal interpolation or smoothing of any kind.
- No vegetation index computation (done in Python later).
- No feature engineering.
- Index computation, cloud gap filling, EMA, and ALL feature engineering
  are handled externally in Python (pandas/numpy) for reproducibility and
  full control over causality.

CLOUD MASKING APPLIED IN GEE (§3.1)
-------------------------------------
SCL classes masked: 0 (no data), 1 (saturated), 2 (dark area / shadows),
3 (cloud shadow), 8 (cloud medium prob.), 9 (cloud high prob.),
10 (thin cirrus), 11 (snow/ice).
Morphological dilation: 1 × 10 m pixel expansion to catch cloud edges.
Masked pixels are set to NaN in the export.

USAGE
-----
1. Install the Earth Engine Python API:
       pip install earthengine-api

2. Authenticate (first run only):
       earthengine authenticate

3. Edit the CONFIGURATION block below (project ID, Drive folder, points CSV).

4. Run:
       python gee_extract.py

   Export tasks are submitted to GEE and run server-side.
   Monitor progress at https://code.earthengine.google.com/tasks

DEPENDENCIES
------------
    earthengine-api >= 0.1.370
    pandas >= 2.0
    (All other operations happen server-side on GEE.)
"""

import ee
import pandas as pd
import time
import math

# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────────────────────────

GEE_PROJECT   = "your-gee-project-id"       # GEE Cloud project ID
DRIVE_FOLDER  = "harvest_gee_exports"        # Destination folder in Google Drive
POINTS_CSV    = "sample_points.csv"          # Local CSV: columns [point_id, lat, lon]

# Temporal range (§2.1). Extend to 2019 if quota allows (§12, Optional Extension).
START_DATE    = "2022-05-01"
END_DATE      = "2025-12-15"
YEARS         = list(range(2022, 2026))      # [2022, 2023, 2024, 2025]

# S2 scene-level pre-filter: keep images with cloud cover below this threshold.
# Per-pixel SCL masking is the real filter; this just avoids downloading
# completely useless near-100%-cloudy images.
S2_MAX_CLOUD_PCT = 40

# Sentinel-1 pass direction (§2.2)
S1_PASS       = "DESCENDING"

# Scale for reduceRegions point extraction (metres).
# 10 m matches native resolution of both S1 and S2.
EXTRACT_SCALE = 10

# ─────────────────────────────────────────────────────────────────
# INITIALISE GEE
# ─────────────────────────────────────────────────────────────────

def init_gee(project: str) -> None:
    """Authenticate and initialise the Earth Engine Python API."""
    try:
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")
    except ee.EEException:
        print("  GEE credentials not found. Running ee.Authenticate() …")
        ee.Authenticate()
        ee.Initialize(project=project)
        print(f"✓ GEE initialised — project: {project}")


# ─────────────────────────────────────────────────────────────────
# LOAD SAMPLE POINTS
# ─────────────────────────────────────────────────────────────────

def load_points(csv_path: str) -> ee.FeatureCollection:
    """
    Read a CSV of sample points and return a GEE FeatureCollection.

    Expected CSV columns: point_id (int), lat (float), lon (float).
    Any extra columns are preserved as Feature properties.
    """
    df = pd.read_csv(csv_path)
    required_cols = {"point_id", "lat", "lon"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"Points CSV must contain columns: {required_cols}. "
            f"Found: {list(df.columns)}"
        )

    features = []
    for _, row in df.iterrows():
        geom  = ee.Geometry.Point([float(row["lon"]), float(row["lat"])])
        props = {str(k): v for k, v in row.items()}
        features.append(ee.Feature(geom, props))

    fc = ee.FeatureCollection(features)
    print(f"✓ Loaded {len(df):,} sample points from '{csv_path}'")
    return fc


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — SCL CLOUD MASKING
# ─────────────────────────────────────────────────────────────────

def build_scl_mask(image: ee.Image) -> ee.Image:
    """
    Build a per-pixel cloud/shadow/snow mask from the SCL band (§3.1).

    SCL classes that are MASKED (set to NaN on export):
        0  — No data
        1  — Saturated / defective
        2  — Dark area pixels (cast shadows, dark soils)
        3  — Cloud shadow
        8  — Cloud medium probability
        9  — Cloud high probability
       10  — Thin cirrus
       11  — Snow / Ice

    SCL classes that are KEPT:
        4  — Vegetation
        5  — Not-vegetated
        6  — Water
        7  — Unclassified

    Morphological dilation (1 pixel / 10 m) is applied to the mask to
    remove contaminated cloud-edge pixels (§3.1, Criticism 11 adjudication).
    """
    scl = image.select("SCL")

    # Build a boolean mask: 1 = valid pixel, 0 = cloudy/shadow/snow
    invalid_classes = [0, 1, 2, 3, 8, 9, 10, 11]
    is_invalid = scl.eq(invalid_classes[0])
    for cls in invalid_classes[1:]:
        is_invalid = is_invalid.Or(scl.eq(cls))

    is_valid = is_invalid.Not()

    # Morphological erosion of the valid mask = dilation of the cloud mask.
    # focal_min with a 1-pixel (10 m) kernel shrinks the valid region by 1 pixel
    # around every cloud edge, effectively discarding contaminated border pixels.
    valid_dilated = is_valid.focal_min(radius=1, kernelType="square", units="pixels")

    return valid_dilated  # 1 = valid, 0 = masked


def mask_s2_clouds(image: ee.Image) -> ee.Image:
    """Apply SCL-based cloud/shadow mask to a Sentinel-2 image."""
    mask = build_scl_mask(image)
    # updateMask sets masked pixels to NaN, which propagates to the CSV export
    return image.updateMask(mask)


# ─────────────────────────────────────────────────────────────────
# SENTINEL-2 — BAND EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

# Raw bands to extract (§2.1).
# NOTE: SCL is NOT in this list — it is used for masking only and then dropped
# so the export contains only the radiometric surface-reflectance bands.
S2_BANDS = ["B2", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]


def extract_s2_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract raw S2 surface-reflectance bands
    for all sample points.

    Returns a FeatureCollection where each Feature = one point × one image date.
    Columns: point_id, lat, lon, date (YYYY-MM-DD), B2, B4, B5, B6, B7, B8,
             B11, B12.  Masked pixels are absent from the export (NaN in CSV).
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD_PCT))
        .select(S2_BANDS + ["SCL"])
        .map(mask_s2_clouds)
        .select(S2_BANDS)   # Drop SCL after masking
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        """reduceRegions over all points for a single image."""
        date_str = image.date().format("YYYY-MM-dd")

        reduced = image.reduceRegions(
            collection  = points,
            reducer     = ee.Reducer.mean(),  # Mean over the ~10 m buffer
            scale       = EXTRACT_SCALE,
            crs         = image.projection()
        )

        # Attach the image acquisition date to every point feature
        return reduced.map(lambda f: f.set("date", date_str))

    # Map over the entire collection → flat FeatureCollection of (point × date) rows
    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# SENTINEL-1 — SAR BACKSCATTER EXTRACTION (one year at a time)
# ─────────────────────────────────────────────────────────────────

S1_BANDS = ["VH", "VV"]


def extract_s1_year(points: ee.FeatureCollection, year: int) -> ee.FeatureCollection:
    """
    For a single calendar year, extract Sentinel-1 GRD VH and VV backscatter
    (in dB) for all sample points.

    Filters applied (§2.2):
      - Instrument mode: IW (Interferometric Wide)
      - Pass direction: DESCENDING (consistent geometry across dates)
      - Bands: VH, VV

    Returns FeatureCollection: point_id, date, VH, VV.
    """
    start = f"{year}-05-01"
    end   = f"{year}-12-15"

    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterDate(start, end)
        .filterBounds(points)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("orbitProperties_pass", S1_PASS))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .select(S1_BANDS)
    )

    def extract_image(image: ee.Image) -> ee.FeatureCollection:
        date_str = image.date().format("YYYY-MM-dd")
        reduced  = image.reduceRegions(
            collection = points,
            reducer    = ee.Reducer.mean(),
            scale      = EXTRACT_SCALE,
        )
        return reduced.map(lambda f: f.set("date", date_str))

    all_rows = collection.map(extract_image).flatten()
    return all_rows


# ─────────────────────────────────────────────────────────────────
# STATIC LAYERS — SRTM + ESA WORLDCOVER (one-time extract)
# ─────────────────────────────────────────────────────────────────

def extract_static_layers(points: ee.FeatureCollection) -> ee.FeatureCollection:
    """
    Extract static (time-invariant) geographic layers for each sample point.

    Layers extracted:
      - SRTM v4.1 @ 30 m: elevation (m), slope (°), aspect (°)
      - ESA WorldCover 2021 @ 10 m: land cover class integer
        (10 = tree cover, 40 = cropland, etc.)

    These are fetched once and merged with the time-series data during the
    Python feature engineering step.
    """
    # SRTM elevation + derived terrain metrics
    srtm      = ee.Image("USGS/SRTMGL1_003")
    elevation = srtm.select("elevation")
    slope     = ee.Terrain.slope(elevation)
    aspect    = ee.Terrain.aspect(elevation)

    terrain = elevation.rename("elevation_m") \
                       .addBands(slope.rename("slope_deg")) \
                       .addBands(aspect.rename("aspect_deg"))

    # ESA WorldCover 2021 (10 m)
    worldcover = (
        ee.ImageCollection("ESA/WorldCover/v200")
        .first()
        .select("Map")
        .rename("worldcover_class")
    )

    static_image = terrain.addBands(worldcover)

    static_fc = static_image.reduceRegions(
        collection = points,
        reducer    = ee.Reducer.mean(),
        scale      = 30,   # SRTM native resolution
    )

    return static_fc


# ─────────────────────────────────────────────────────────────────
# COLUMN CLEANUP — SELECT ONLY REQUIRED COLUMNS FOR EXPORT
# ─────────────────────────────────────────────────────────────────

def select_s2_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S2 CSV."""
    keep = ["point_id", "lat", "lon"] + S2_BANDS + ["date"]
    return fc.select(keep)


def select_s1_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the final S1 CSV."""
    keep = ["point_id", "lat", "lon"] + S1_BANDS + ["date"]
    return fc.select(keep)


def select_static_columns(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only the columns needed in the static CSV."""
    keep = ["point_id", "lat", "lon",
            "elevation_m", "slope_deg", "aspect_deg", "worldcover_class"]
    return fc.select(keep)


# ─────────────────────────────────────────────────────────────────
# EXPORT HELPERS
# ─────────────────────────────────────────────────────────────────

def export_to_drive(
    fc: ee.FeatureCollection,
    description: str,
    folder: str,
    filename: str,
) -> ee.batch.Task:
    """
    Submit a GEE Export.table.toDrive task for a FeatureCollection.

    Returns the Task object (already started). Monitor at
    https://code.earthengine.google.com/tasks
    """
    task = ee.batch.Export.table.toDrive(
        collection    = fc,
        description   = description,
        folder        = folder,
        fileNamePrefix= filename,
        fileFormat    = "CSV",
    )
    task.start()
    print(f"  → Task submitted: '{description}'  (filename: {filename}.csv)")
    return task


def wait_for_tasks(tasks: list, poll_interval_s: int = 30) -> None:
    """
    Poll all submitted tasks until they complete or fail.
    Optional — you can also just let them run and monitor on the GEE Tasks page.
    """
    print("\n⏳  Polling task status (Ctrl+C to stop polling without cancelling tasks) …")
    remaining = {t.id: t for t in tasks}

    while remaining:
        time.sleep(poll_interval_s)
        done = []
        for tid, task in remaining.items():
            status = task.status()
            state  = status["state"]
            name   = status.get("description", tid)
            if state in ("COMPLETED", "FAILED", "CANCELLED"):
                icon = "✓" if state == "COMPLETED" else "✗"
                print(f"  {icon} [{state}] {name}")
                done.append(tid)
        for tid in done:
            del remaining[tid]

    print("✓ All tasks finished.")


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def main() -> None:
    # ── 1. Initialise ────────────────────────────────────────────
    init_gee(GEE_PROJECT)

    # ── 2. Load points ───────────────────────────────────────────
    points = load_points(POINTS_CSV)

    submitted_tasks = []

    # ── 3. Sentinel-2 export — one task per year ─────────────────
    print("\n── Sentinel-2 raw band extraction ──────────────────────────")
    for year in YEARS:
        print(f"  Processing S2 year {year} …")
        fc       = extract_s2_year(points, year)
        fc_clean = select_s2_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S2_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel2_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 4. Sentinel-1 export — one task per year ─────────────────
    print("\n── Sentinel-1 SAR backscatter extraction ───────────────────")
    for year in YEARS:
        print(f"  Processing S1 year {year} …")
        fc       = extract_s1_year(points, year)
        fc_clean = select_s1_columns(fc)
        task     = export_to_drive(
            fc          = fc_clean,
            description = f"S1_raw_{year}",
            folder      = DRIVE_FOLDER,
            filename    = f"sentinel1_raw_{year}",
        )
        submitted_tasks.append(task)

    # ── 5. Static layers export — one-time ───────────────────────
    print("\n── Static layers (SRTM + WorldCover) ───────────────────────")
    fc_static  = extract_static_layers(points)
    fc_static_clean = select_static_columns(fc_static)
    task_static = export_to_drive(
        fc          = fc_static_clean,
        description = "static_layers",
        folder      = DRIVE_FOLDER,
        filename    = "static_layers",
    )
    submitted_tasks.append(task_static)

    # ── 6. Summary ───────────────────────────────────────────────
    total = len(submitted_tasks)
    print(f"\n✓ {total} export tasks submitted to GEE.")
    print(f"  Files will appear in Google Drive → '{DRIVE_FOLDER}/' once complete.")
    print("  Monitor progress at: https://code.earthengine.google.com/tasks\n")

    print("Expected output files:")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel2_raw_{year}.csv")
    for year in YEARS:
        print(f"  {DRIVE_FOLDER}/sentinel1_raw_{year}.csv")
    print(f"  {DRIVE_FOLDER}/static_layers.csv")

    # ── 7. Optional: block and poll until all tasks finish ────────
    # Uncomment the line below if you want the script to wait and
    # print live status updates.  Otherwise tasks run in background.
    # wait_for_tasks(submitted_tasks, poll_interval_s=30)


if __name__ == "__main__":
    main()

✓ GEE initialised — project: your-gee-project-id


FileNotFoundError: [Errno 2] No such file or directory: 'sample_points.csv'